<a href="https://colab.research.google.com/github/AzizulHakim00/Glaucomma/blob/main/Glaucomma_RimGraphDG_V45_ORIGA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RimGraph-DG V4.5 — mask-audited ORIGA-first run
Use a **T4 GPU**. This run validates decoded OD/OC supervision before model training, reuses the completed V4.4 ORIGA baseline only if split/config signatures match exactly, and trains RimGraph fresh.

In [1]:
GLAUCOMMA_OVERRIDES = {
    'manual_data_dir': '',
    'fast_dev_run': False,
    'run_name': 'paper_run_v45',
    'code_revision': 'rimgraph-dg-v4.5-20260809',
    'seeds': [2029],
    'fold_targets': ['ORIGA'],
    'run_global_baseline': True,
    'run_full_model': True,
    'baseline_reuse_run': 'paper_run_v44',
    'run_optuna': False,
    'image_size': 320,
    'batch_size': 2,
    'grad_accum': 2,
    'fpn_dim': 128,
    'gradient_checkpointing': True,
    'baseline_epochs': 12,
    'full_epochs': 30,
    'seg_warmup_epochs': 5,
    'anatomy_warmup_epochs': 10,
    'num_workers': 0,
    'n_visual_examples': 2,
    'resume': True,
}

import hashlib, json, traceback, urllib.request
from pathlib import Path
import torch

print('=== V4.5 LAUNCHER GPU CHECK ===', flush=True)
print('PyTorch:', torch.__version__, flush=True)
print('CUDA available:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('T4 GPU is not active. Colab: Runtime > Change runtime type > T4 GPU, reconnect, then rerun.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('================================', flush=True)

COMMIT = 'd71c11be1a13a25ed0352cdde01453c17b234429'
EXPECTED_RAW_SHA256 = '46ba27c7446662460456bc2bab186729c0df1b3e76533ce44f208150208335e2'
ROOT = f'https://raw.githubusercontent.com/AzizulHakim00/Glaucomma/{COMMIT}'
parts = [f'v4_parts/part_{i:02d}.py' for i in range(7)]
raw_code = '\n'.join(urllib.request.urlopen(f'{ROOT}/{name}').read().decode('utf-8') for name in parts)
actual_raw = hashlib.sha256(raw_code.encode('utf-8')).hexdigest()
assert actual_raw == EXPECTED_RAW_SHA256, f'Raw runner integrity check failed: {actual_raw}'

patch_specs = [
    ('runner_patch_v41.py', 'apply_v41'),
    ('runner_patch_v42.py', 'apply_v42'),
    ('runner_patch_v43.py', 'apply_v43'),
    ('runner_patch_v43_autograd.py', 'apply_v43_autograd'),
    ('runner_patch_v44_runtime.py', 'apply_v44_runtime'),
    ('runner_patch_v45_masks.py', 'apply_v45_masks'),
    ('runner_patch_v45_lowlabels.py', 'apply_v45_lowlabels'),
]
code = raw_code
for patch_name, function_name in patch_specs:
    print(f'[LAUNCHER] applying {patch_name}', flush=True)
    source = urllib.request.urlopen(f'{ROOT}/{patch_name}').read().decode('utf-8')
    namespace = {}
    exec(compile(source, patch_name, 'exec'), namespace, namespace)
    code = namespace[function_name](code)
compile(code, 'rimgraph_dg_v45_origa.py', 'exec')
print('[LAUNCHER] V4.5 assembly PASSED', flush=True)

try:
    exec(code, globals(), globals())
    completion = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45/RUN_COMPLETED.json')
    if not completion.exists():
        raise RuntimeError('Runner returned without RUN_COMPLETED.json; treating this as failure.')
    print('\n✅ V4.5 VERIFIED COMPLETION:', completion, flush=True)
except BaseException:
    trace = traceback.format_exc()
    print('\n=== V4.5 FAILURE TRACEBACK ===', flush=True)
    print(trace, flush=True)
    try:
        Path('/content/RimGraph_V45_FAILURE_TRACEBACK.txt').write_text(trace, encoding='utf-8')
    except Exception:
        pass
    try:
        failure_dir = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45')
        failure_dir.mkdir(parents=True, exist_ok=True)
        (failure_dir / 'FAILURE_TRACEBACK.txt').write_text(trace, encoding='utf-8')
        (failure_dir / 'FAILURE_STATUS.json').write_text(json.dumps({'status': 'failed', 'traceback_file': str(failure_dir / 'FAILURE_TRACEBACK.txt')}, indent=2), encoding='utf-8')
    except Exception as drive_error:
        print(f'Could not persist failure to Drive: {drive_error}', flush=True)
    raise


=== V4.5 LAUNCHER GPU CHECK ===
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
[LAUNCHER] applying runner_patch_v41.py
[LAUNCHER] applying runner_patch_v42.py
[LAUNCHER] applying runner_patch_v43.py
[LAUNCHER] applying runner_patch_v43_autograd.py
[LAUNCHER] applying runner_patch_v44_runtime.py
[LAUNCHER] applying runner_patch_v45_masks.py
[LAUNCHER] applying runner_patch_v45_lowlabels.py
[LAUNCHER] V4.5 assembly PASSED
Mounted at /content/drive
Resolved Colab output: /content/Glaucomma_runs/paper_run_v45
Resolved Drive output: /content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45
Drive write verification: PASSED

=== RIMGRAPH V4.4 RUNTIME ===
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB



## RimGraph-DG V4 configuration

,setting,value
0,kaggle_dataset,arnavjain1/glaucoma-datasets
1,manual_data_dir,
2,sources,"['ORIGA', 'REFUGE', 'G1020']"
3,fold_targets,['ORIGA']
4,image_size,320
5,num_workers,0
6,canonicalize_laterality,True
7,exclude_cross_source_duplicates,True
8,project_name,RimGraph_DG_V4
9,run_name,paper_run_v45


## Downloading or locating Kaggle dataset

100%|██████████| 5.55G/5.55G [05:18<00:00, 18.7MB/s]

Extracting files...


... (19 outputs left out)

## Baseline versus full model

,held_out_source,seed,baseline_auroc,full_auroc,delta_auroc,baseline_auprc,full_auprc,delta_auprc
0,ORIGA,2029,0.7627,0.7765,0.0138,0.5216,0.4926,-0.0289


# RimGraph-DG V4 Scientific Diagnostic Report

- Global baseline mean external AUROC: **0.7627**
- Global baseline worst-domain AUROC: **0.7627**
- RimGraph V4 mean external AUROC: **0.7765**
- RimGraph V4 worst-domain AUROC: **0.7765**
- Mean disc Dice: **0.9495**
- Mean cup Dice: **0.7343**
- Mean ECE: **0.1221**

## Interpretation
- Full-model discrimination is close to the baseline; novelty must be justified through calibration, robustness, segmentation, and ablations rather than AUROC alone.
- Cup segmentation is too weak for reliable structural graph interpretation. Improve masks/decoder before publication claims.
- Severe domain-generalization weakness remains; the current result is not journal-ready.
- Calibration is poor; probabilities should not be presented as clinically reliable.

## Required publication checks
- Run at least three seeds for every held-out domain.
- Report bootstrap confidence intervals and baseline-versus-full deltas.
- Polar-sector XAI is image-oriented when laterality is unknown; do not label it as clinical clock-hour anatomy.
- gate_context is a latent fusion variable, not a supervised image-quality score.
- Manually inspect segmentation and XAI failures.
- Add one completely untouched external dataset for a stronger Q1/Q2 claim.


✅ V4.5 VERIFIED COMPLETION: /content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45/RUN_COMPLETED.json
